# SPA Chatbot - LangChain Prompt Pipeline

**Exercise objective:** given a conversation excerpt between a SPA chatbot and a user, design a prompting pipeline in LangChain that produces a precise, well-structured answer to the user's question.

**Conversation context provided:**
- The SPA offers a "sensory experience" package at no extra cost.
- The chatbot described the package: essential oils, warm massages with hot stones, warm bath, ambient sounds and dim lighting.
- **User question:** *"A cosa servono gli oli essenziali?"* (What are essential oils for?)

**Stack:** all models run locally via [Ollama](https://ollama.com) - no API keys required.

| Model | Role |
|-------|------|
| `llama3.2` | Strategies 1 and 3 (persona + reasoning) |
| `mistral` | Strategy 2 (demonstrates provider swap) |

**Pipeline strategies:**

| # | Technique | Goal |
|---|-----------|------|
| 1 | System prompt + few-shot | Establish chatbot persona and response style |
| 2 | RAG-style grounding | Ground answer in a structured knowledge base |
| 3 | Chain-of-thought + self-critique | Force reasoning before answering, then refine |

## Setup

Ollama must be running before executing this notebook.
If it is not already active as a background service, start it from the terminal:

```bash
ollama serve
```

Required models (already downloaded):
```bash
ollama pull llama3.2
ollama pull mistral
```

In [13]:
# !pip install langchain langchain-ollama

import subprocess

# Verify Ollama is reachable and both models are available
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

assert "llama3.2" in result.stdout, "llama3.2 not found - run: ollama pull llama3.2"
assert "mistral"  in result.stdout, "mistral not found - run: ollama pull mistral"
print("Both models available. Ready.")

NAME               ID              SIZE      MODIFIED       
mistral:latest     6577803aa9a0    4.4 GB    11 minutes ago    
llama3.2:latest    a80c4f17acd5    2.0 GB    12 minutes ago    

Both models available. Ready.


In [14]:
# ── Shared context: the conversation history provided in the exercise ──────────

CONVERSATION_HISTORY = """
Chatbot: L'esperienza nella SPA include il pacchetto "esperienza sensoriale". Non è previsto alcun costo aggiuntivo.
Chatbot: L'esperienza sensoriale in SPA è un viaggio di relax totale: ti immergi in un ambiente rilassante,
avvolto da aromi di oli essenziali, mentre suoni naturali e luce soffusa creano un'atmosfera di pace.
Include un massaggio con oli caldi e pietre levigate, che sciolgono le tensioni muscolari,
accompagnato da un bagno caldo e avvolgente. Tutti i sensi vengono stimolati per favorire il massimo benessere.
""".strip()

USER_QUESTION = "A cosa servono gli oli essenziali?"

# ── SPA knowledge base: structured reference used by Strategy 2 ───────────────
SPA_KNOWLEDGE_BASE = """
ELEMENTI DELL'ESPERIENZA SENSORIALE:

Aromi (oli essenziali): Gli oli essenziali, come lavanda o eucalipto, aiutano a rilassare la mente
e migliorare l'umore. Inalando questi profumi, si riduce lo stress e si promuove il rilassamento.

Massaggi: Sciolgono le tensioni muscolari, migliorano la circolazione sanguigna e riducono
il dolore o la rigidità. Il cliente si sente più leggero e disteso.

Calore (sauna, bagno turco, pietre calde): Il calore penetra in profondità nei muscoli,
aumentando la circolazione e favorendo la disintossicazione del corpo. Aiuta anche a rilassare la mente.

Acqua (idromassaggio, vasche): Il movimento dell'acqua massaggia il corpo, riducendo lo stress
fisico e migliorando il flusso linfatico, essenziale per la depurazione.

Suoni e luci soffuse: Creano un ambiente di calma, riducendo la stimolazione eccessiva
della mente e permettendo una sensazione di pace e tranquillità.
""".strip()

print(f"User question: '{USER_QUESTION}'")

User question: 'A cosa servono gli oli essenziali?'


---
## Strategy 1 - System Prompt + Few-Shot Examples

**Technique:** combine a system-level persona definition with two few-shot examples that demonstrate the expected answer format. The model infers the style (tone, structure, detail level) from the examples without explicit formatting instructions.

**Why this works:**
- The system prompt anchors the model's role and constraints, preventing off-topic answers.
- Few-shot examples are the most reliable way to specify output format: showing is more precise than describing.
- Conversation history is passed as a variable so the same template is reusable across sessions.

**LangChain components:**
- `ChatOllama` - wraps the local Ollama server with the same interface as `ChatAnthropic` / `ChatOpenAI`
- `ChatPromptTemplate.from_messages` - structures a multi-turn prompt
- `StrOutputParser` - extracts text from the model response object
- `|` operator (LCEL) - chains components into a pipeline

In [15]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Model ─────────────────────────────────────────────────────────────────────
llm_llama = ChatOllama(
    model="llama3.2",
    temperature=0.3,  # low temperature → focused, consistent answers
)

# ── Prompt: system + few-shot examples + conversation history + question ───────
prompt_s1 = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly and knowledgeable assistant for a luxury SPA. "
     "You answer guests' questions in Italian, clearly and warmly. "
     "When a guest asks about a SPA element, explain its specific benefit concisely. "
     "Do not invent services or prices not mentioned in the conversation."),

    # Few-shot example 1
    ("human",  "A cosa serve il massaggio con le pietre?"),
    ("assistant",
     "Le pietre levigate vengono scaldate e appoggiate sui punti di tensione del corpo. "
     "Il calore penetra in profondità nei muscoli, scioglie i nodi e migliora la circolazione, "
     "lasciandoti una sensazione di leggerezza e benessere duraturo."),

    # Few-shot example 2
    ("human",  "Il bagno caldo è incluso nel pacchetto?"),
    ("assistant",
     "Sì, il bagno caldo è parte dell'esperienza sensoriale, inclusa senza costi aggiuntivi. "
     "L'acqua calda favorisce il rilassamento muscolare e prepara il corpo ai trattamenti successivi."),

    # Actual conversation history + real user question
    ("human",  "[Contesto della conversazione]:\n{history}\n\nDomanda: {question}"),
])

# ── Pipeline (LCEL) ────────────────────────────────────────────────────────────
chain_s1 = prompt_s1 | llm_llama | StrOutputParser()

response_s1 = chain_s1.invoke({
    "history":  CONVERSATION_HISTORY,
    "question": USER_QUESTION,
})

print("=" * 60)
print("STRATEGY 1 - System Prompt + Few-Shot  [llama3.2]")
print("=" * 60)
print(response_s1)

STRATEGY 1 - System Prompt + Few-Shot  [llama3.2]
Gli oli essenziali sono utilizzati per creare un'atmosfera rilassante e promuovere la calma. Sono stati scelti per il loro effetto terapeutico, che favorisce la relaxazione e riduce lo stress. Durante l'esperienza sensoriale, gli oli essenziali vengono utilizzati per creare un ambiente di benessere, aiutando a calmare la mente e a rilassare il corpo.


---
## Strategy 2 - RAG-style Grounding  (provider swap: Mistral)

**Technique:** instead of relying on the model's parametric knowledge, explicitly inject a **structured knowledge base** into the prompt. The model is instructed to answer *only* from that document.

**Why this works:**
- Grounding eliminates hallucination: the model cannot invent benefits that are not in the KB.
- The `knowledge_base` variable separates content from conversation logic - updating the SPA's services only requires updating the KB, not the prompt.
- In a production RAG system, this variable would be filled by a vector retriever (e.g. ChromaDB + embeddings), not hardcoded. The prompt template shape stays identical.

**Provider swap:** this strategy uses `mistral` to demonstrate that switching models in LangChain requires changing only the model object. The prompt template, output parser, and chain composition are unchanged. This is the key value of LangChain's provider-agnostic interface.

In [16]:
# One-line provider swap: same interface, different local model
llm_mistral = ChatOllama(
    model="mistral",
    temperature=0.2,  # lower → more faithful to the KB
)

# ── Prompt: grounded on knowledge base ────────────────────────────────────────
prompt_s2 = ChatPromptTemplate.from_messages([
    ("system",
     "You are a SPA assistant. Answer the guest's question using ONLY the information "
     "provided in the KNOWLEDGE BASE section below. "
     "If the answer is not in the knowledge base, say so politely. "
     "Respond in Italian. Be concise and clear."),
    ("human",
     "KNOWLEDGE BASE:\n{knowledge_base}\n\n"
     "CONVERSATION SO FAR:\n{history}\n\n"
     "GUEST QUESTION: {question}"),
])

# Same chain structure as Strategy 1 - only llm_mistral changed
chain_s2 = prompt_s2 | llm_mistral | StrOutputParser()

response_s2 = chain_s2.invoke({
    "knowledge_base": SPA_KNOWLEDGE_BASE,
    "history":        CONVERSATION_HISTORY,
    "question":       USER_QUESTION,
})

print("=" * 60)
print("STRATEGY 2 - RAG-style Grounding  [mistral]")
print("=" * 60)
print(response_s2)

STRATEGY 2 - RAG-style Grounding  [mistral]
 Gli oli essenziali aiutano a rilassare la mente, riducendo lo stress e promuovendo il rilassamento.


---
## Strategy 3 - Chain-of-Thought + Self-Critique (two-step chain)

**Technique:** split the task into two sequential LLM calls connected via LCEL:

1. **Step A - Chain-of-Thought (CoT):** ask the model to reason step by step before producing an answer. CoT forces explicit decomposition of the problem, reducing errors caused by premature commitment to a wrong interpretation.
2. **Step B - Self-critique and refinement:** the draft from Step A is passed to a second call that critiques and rewrites it for tone, completeness, and conciseness - acting as an editorial layer.

**Why two steps instead of one complex prompt:**
- Each call is simpler and focused, reducing the risk of conflicting instructions in one large prompt.
- The intermediate reasoning is inspectable and debuggable as a standalone output.
- The refinement step converts a correct-but-verbose CoT answer into a clean, guest-facing response without re-running the reasoning.

**LangChain components:**
- `RunnablePassthrough.assign` - runs a sub-chain and adds its output as a new key to the input dict, without discarding existing keys
- The result is piped directly into the refinement prompt, which needs both `question` (from the original input) and `draft` (from Step A)

In [17]:
from langchain_core.runnables import RunnablePassthrough

# Both steps use llama3.2; temperature differs by role
llm_cot    = ChatOllama(model="llama3.2", temperature=0.5)  # more creative for reasoning
llm_refine = ChatOllama(model="llama3.2", temperature=0.1)  # conservative for editing

# ── Step A: Chain-of-Thought reasoning ────────────────────────────────────────
prompt_cot = ChatPromptTemplate.from_messages([
    ("system",
     "You are a SPA wellness expert. Think step by step before answering. "
     "First, identify every benefit of the element the guest asked about. "
     "Then, connect those benefits to the sensory experience described in the conversation. "
     "Finally, draft a clear answer in Italian."),
    ("human",
     "Conversation so far:\n{history}\n\n"
     "Guest question: {question}\n\n"
     "Think step by step, then provide your answer."),
])

# ── Step B: Self-critique and refinement ──────────────────────────────────────
prompt_refine = ChatPromptTemplate.from_messages([
    ("system",
     "You are an editor for a luxury SPA chatbot. "
     "You receive a draft answer and must refine it: "
     "remove any internal reasoning visible to the guest, "
     "ensure the tone is warm and professional, "
     "keep the answer concise (max 4 sentences), "
     "and make sure it directly addresses the guest's question. "
     "Output only the final refined answer in Italian."),
    ("human",
     "Original guest question: {question}\n\n"
     "Draft answer to refine:\n{draft}"),
])

# ── Two-step LCEL chain ────────────────────────────────────────────────────────
# RunnablePassthrough.assign(draft=...) runs chain_cot and adds 'draft' to the
# existing dict, so the next step receives {history, question, draft}.
chain_cot = prompt_cot | llm_cot | StrOutputParser()

chain_s3 = (
    RunnablePassthrough.assign(draft=chain_cot)
    | prompt_refine
    | llm_refine
    | StrOutputParser()
)

input_dict = {"history": CONVERSATION_HISTORY, "question": USER_QUESTION}

# Capture intermediate CoT output separately for inspection
draft_answer = chain_cot.invoke(input_dict)
final_answer = chain_s3.invoke(input_dict)

print("=" * 60)
print("STRATEGY 3 - Step A: Chain-of-Thought reasoning  [llama3.2]")
print("=" * 60)
print(draft_answer)

print()
print("=" * 60)
print("STRATEGY 3 - Step B: After self-critique refinement  [llama3.2]")
print("=" * 60)
print(final_answer)

STRATEGY 3 - Step A: Chain-of-Thought reasoning  [llama3.2]
Per rispondere alla domanda del nostro ospite, dobbiamo identificare i benefici degli oli essenziali e collegarli all'esperienza sensoriale descritta nell'ambiente della SPA.

Gli oli essenziali sono noti per le loro proprietà terapeutiche e benefiche per la salute. Alcuni dei principali benefici includono:

* Riduzione dello stress e dell'ansia grazie alle loro proprietà calmanti
* Miglioramento del benessere emotivo e mentale
* Rilassamento muscolare e riduzione della tensione
* Improvazione della qualità del sonno
* Stimolazione del sistema immunitario

Ora, collegando questi benefici agli oli essenziali nell'ambiente della SPA, possiamo capire come siano utilizzati per creare un'esperienza di relax totale.

Gli oli essenziali vengono utilizzati per creare un'atmosfera rilassante e pacifica, che aiuta a ridurre lo stress e l'ansia. I loro aromi calmi e soffusi contribuiscono a creare un senso di benessere e tranquillità, fa

---
## Comparison and Analysis

In [18]:
import textwrap

strategies = {
    "S1 - Few-Shot        [llama3.2]": response_s1,
    "S2 - RAG-style       [mistral] ": response_s2,
    "S3 - CoT + Critique  [llama3.2]": final_answer,
}

for name, resp in strategies.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")
    print(textwrap.fill(resp.strip(), width=70))
    print(f"  [word count: {len(resp.split())}]")


────────────────────────────────────────────────────────────
  S1 - Few-Shot        [llama3.2]
────────────────────────────────────────────────────────────
Gli oli essenziali sono utilizzati per creare un'atmosfera rilassante
e promuovere la calma. Sono stati scelti per il loro effetto
terapeutico, che favorisce la relaxazione e riduce lo stress. Durante
l'esperienza sensoriale, gli oli essenziali vengono utilizzati per
creare un ambiente di benessere, aiutando a calmare la mente e a
rilassare il corpo.
  [word count: 53]

────────────────────────────────────────────────────────────
  S2 - RAG-style       [mistral] 
────────────────────────────────────────────────────────────
Gli oli essenziali aiutano a rilassare la mente, riducendo lo stress e
promuovendo il rilassamento.
  [word count: 15]

────────────────────────────────────────────────────────────
  S3 - CoT + Critique  [llama3.2]
────────────────────────────────────────────────────────────
Risposta rifinita: Gli oli essenziali 

---
## Critical Analysis

### What each strategy optimises for

**Strategy 1 - System Prompt + Few-Shot**  
The few-shot examples steer tone and format without listing explicit rules. The model infers the response pattern from demonstration, which is more reliable than natural language style instructions. The tradeoff: the examples add tokens to every request and must be chosen carefully - poorly chosen examples can misdirect the model more than no examples at all. Best when response *style* matters more than strict factual control.

**Strategy 2 - RAG-style Grounding**  
Injecting the structured KB makes the response fully auditable: every claim in the output should trace back to a sentence in the document. This eliminates hallucination and makes the system updatable without touching the prompt logic. The constraint is that the model can only answer what is in the KB. In production this variable would be filled by a vector retriever (e.g. ChromaDB lookup), not a hardcoded string - but the prompt template shape is production-ready as written.

**Strategy 3 - CoT + Self-Critique**  
Forcing explicit reasoning before answering reduces errors from premature commitment to a wrong interpretation. The refinement step separates *thinking* from *presentation*, producing a draft that is correct-but-verbose and a final answer that is polished. Cost: latency and double the token usage. Justified when the question is complex or ambiguous. For a simple factual question like this one the improvement over S1/S2 is marginal - CoT shines on multi-step reasoning tasks.

### Local models vs cloud APIs: practical notes

Running Ollama on an M3 Pro gives Metal-accelerated inference - llama3.2 (3B) and Mistral (7B) are fast enough for interactive use. The main differences from cloud models:
- **No latency for API calls** - everything is local
- **Instruction-following is less reliable** than GPT-4 / Claude for complex multi-step prompts
- **Temperature sensitivity differs** - local models often need slightly higher temperatures to avoid repetitive outputs
- **Context window** - llama3.2 supports 128k tokens; mistral 32k

### Recommended strategy for production
For a SPA chatbot:
- Use **Strategy 2** (RAG-style) as the foundation - factual accuracy is non-negotiable in a hospitality context.
- Layer **Strategy 1**'s few-shot examples on top to control tone and format.
- Reserve **Strategy 3** (CoT) for complex queries: complaint handling, multi-service comparisons, ambiguous requests.